# 🚀  How to Build & Test Your Streamlit App in Colab
This notebook is a template for developing and testing your Streamlit app. You will use this to write your code, test it live, and then download the final files to upload to GitHub.



<img src="https://raw.githubusercontent.com/sasrwt/homequity/main/streamlitappflow.png" width="500">

### **Step 1: Install All Libraries**
Run the code cell below. This installs Streamlit, the libraries your app needs (pandas, scikit-learn), and localtunnel (creates a temporary public URL that "tunnels" to the hmeq app running insidColab notebook. This makes our app accessible in any web browser.).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install all required libraries
!pip install streamlit pandas scikit-learn -q
!npm install localtunnel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.9 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸npm notice
npm notice New major version of npm available! 10.8.2 -> 11.6.4
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.6.4
npm notice To update run: npm install -g npm@11.6.4
npm notice
⠸

### **Step 2: Write Your App Code (`app.py`)**

https://docs.streamlit.io/develop/api-reference/widgets

Run the cell below.  
The `%%writefile` command will take all the code in the cell and save it as a new file named **`app.py`**.

This is where you will do your main work. You need to:

← **Update the `st.slider` and `st.selectbox` widgets** to match the input columns your model requires.

← **Update the `pd.DataFrame`** so its columns match the widgets you created.

← **Change the file path** so it correctly points to the location of your uploaded **`.pkl`** model file.


In [ ]:
%%writefile hmeqapp.py

# -*- coding: utf-8 -*-
import streamlit as st
import pickle
import pandas as pd
import sklearn  # This is needed for the pickle file to load!

# Load the trained model
# --- Put the Model in Drive First---
with open("/content/drive/MyDrive/FinalProj458/my_model.pkl", "rb") as file:
    model = pickle.load(file)

# Title for the app
st.markdown(
    "<h1 style='text-align: center; background-color: #e6f7ff; padding: 10px; color: #004080;'><b>Loan Approval Prediction</b></h1>",
    unsafe_allow_html=True
)

# Numeric inputs
st.header("Enter Applicant's Financial Details and Demographics")

# Input fields for numeric values
granted_loan = st.slider("Granted Loan Amount", min_value=1000, max_value=500000, step=1000, value=50000)
requested_loan = st.slider("Requested Loan Amount", min_value=1000, max_value=500000, step=1000, value=50000)
fico_score = st.slider("FICO Score", min_value=300, max_value=850, step=1, value=650)
monthly_gross_income = st.slider("Monthly Gross Income", min_value=0, max_value=20000, step=100, value=5000)
monthly_housing_payment = st.slider("Monthly Housing Payment", min_value=0, max_value=10000, step=50, value=1500)

# Binary/Categorical inputs
ever_bankrupt = st.checkbox("Ever Bankrupt or Foreclosed?")

reason_options = ['credit_card_refinancing', 'debt_conslidation', 'home_improvement', 'major_purchase', 'other']
reason = st.selectbox("Reason for Loan", reason_options)

fico_group_options = ['poor', 'fair', 'good', 'very_good']
fico_group = st.selectbox("FICO Score Group", fico_group_options, index=1) # Default to 'fair'

employment_status_options = ['full_time', 'part_time', 'unemployed']
employment_status = st.selectbox("Employment Status", employment_status_options, index=0) # Default to 'full_time'

employment_sector_options = [
    'communication_services', 'consumer_discretionary', 'consumer_staples',
    'energy', 'financials', 'health_care', 'industrials',
    'information_technology', 'materials', 'real_estate', 'utilities', 'other_sector'
]
employment_sector = st.selectbox("Employment Sector", employment_sector_options, index=7) # Default to 'information_technology'

lender_options = ['A', 'B', 'C']
lender = st.selectbox("Preferred Lender", lender_options, index=0) # Default to 'A'


# Create the input data as a DataFrame
# Note: These columns are the "original" features before one-hot encoding
input_data = pd.DataFrame({
    "Granted_Loan_Amount": [granted_loan],
    "Requested_Loan_Amount": [requested_loan],
    "FICO_score": [fico_score],
    "Monthly_Gross_Income": [monthly_gross_income],
    "Monthly_Housing_Payment": [monthly_housing_payment],
    "Ever_Bankrupt_or_Foreclose": [1 if ever_bankrupt else 0],
    "Reason": [reason],
    "Fico_Score_group": [fico_group],
    "Employment_Status": [employment_status],
    "Employment_Sector": [employment_sector],
    "Lender": [lender]
})

# --- Prepare Data for Prediction ---
# 1. One-hot encode the user's input.
input_data_encoded = pd.get_dummies(input_data, columns=[
    'Reason', 'Fico_Score_group', 'Employment_Status', 'Employment_Sector', 'Lender'
])

# 2. Add any "missing" columns the model expects (fill with 0).
# This step is crucial if the user selects a category that wasn't in the training data,
# or if get_dummies doesn't produce all expected OHE columns for a given run.
for col_name in model.feature_names_in_:
    if col_name not in input_data_encoded.columns:
        input_data_encoded[col_name] = 0

# 3. Reorder/filter columns to exactly match the model's training data.
input_data_encoded = input_data_encoded[model.feature_names_in_]

# Predict button
if st.button("Predict Loan Outcome"):
    # Predict using the loaded model
    prediction = model.predict(input_data_encoded)[0]

    # Display result
    if prediction == 1:
        st.error("Prediction: **Likely to Default (Bad Loan)** 🚫")
        st.write("This applicant has a higher probability of defaulting on the loan.")
    else:
        st.success("Prediction: **Low Risk (Good Loan)** 💲")
        st.write("This applicant has a lower probability of defaulting on the loan.")

Writing hmeqapp.py


### **Step 3: Write Your Requirements File**

Need this file in GitHub.  
It tells **Streamlit Cloud** which libraries need to be installed.

Run the cell below to create **`requirements.txt`**.

← **Important:** If your model uses additional libraries (such as `xgboost`, `lightgbm`, `catboost`, etc.),  
you **must manually add them** to the list before deploying!


In [ ]:
%%writefile requirements.txt
streamlit
pandas
scikit-learn
numpy

Writing requirements.txt


### **Step 4: Test Your App Live!**

Run the code cell below.  
It will start your **Streamlit app** in the background and then give you a **public URL** to test it.

← Click the URL (it will look something like `https://some-random-words.loca.lt`).

← When the new page opens, click the **"Click to Continue"** button.

← Your app will appear! **Test it thoroughly** to make sure all inputs, predictions, and outputs work as expected.


In [ ]:
# Run the app in the background
!streamlit run hmeqapp.py.py &>/dev/null&

# Get the password (your IP) and print it.
# The "&" runs it in the background while the tunnel starts.
print("Fetching your tunnel password...")
!curl https://loca.lt/mytunnelpassword &

# Start the tunnel. This command will keep the cell running and give you the URL.
!npx localtunnel --port 8501

Fetching your tunnel password...
34.80.120.53⠙⠹your url is: https://free-maps-call.loca.lt
/content/node_modules/localtunnel/bin/lt.js:81
    throw err;
    ^

Error: connection refused: localtunnel.me:31039 (check your firewall settings)
    at Socket.<anonymous> (/content/node_modules/localtunnel/lib/TunnelCluster.js:52:11)
    at Socket.emit (node:events:524:28)
    at emitErrorNT (node:internal/streams/destroy:169:8)
    at emitErrorCloseNT (node:internal/streams/destroy:128:3)
    at process.processTicksAndRejections (node:internal/process/task_queues:82:21)

Node.js v20.19.0
⠙

## Kick the Can and Move On


That TypeError: Failed to fetch is a browser-side error. It means your browser (the app's frontend) tried to download its own JavaScript files from the localtunnel URL, but the tunnel was so slow or unstable that the request failed.

This confirms our theory: your app is taking too long to load the model, and the localtunnel connection is collapsing or timing out before the app can become stable.

💡 The Real Problem
The "Colab + localtunnel" workflow is just a testing tool. It is famously unstable and difficult, as you've seen.

We have been using this flaky, complicated method (managing passwords, running multiple cells, curl commands) just to test the app. But this test is now causing more problems than it's worth.

The good news is we don't need it anymore. We have already learned everything we need from this Colab experiment:

We have the final, working hmeqapp.py code.

We have the final, working requirements.txt file (after fixing the sklearn typo).

We know the hmeq_model.pkl file works.

The goal was never to use the Colab app. The goal was to get the files ready for a real deployment. That mission is accomplished.

### **Step 6: Download Your Files for GitHub**

If your app works, you're ready for deployment.

← Go to the **Files** sidebar on the left.

← Find **`app.py`**. Right-click it and select **Download**.

← Find **`requirements.txt`**. Right-click it and select **Download**.

You are now ready to upload **`app.py`**, **`requirements.txt`**, and your **`.pkl` model file** to a new GitHub repository for deployment.

---

### **Final GitHub Step (Important!)**

Before deploying, edit your **`app.py`** file **one last time** (either in GitHub or in a local editor):

← **Remove the folder path** from your `open()` command.  
It should look like this:

```python
open("your_model_name.pkl", "rb")
